# Text emotion recognition on MELD (Colab)

1. **Runtime → Change runtime type → GPU** (T4 is enough; L4/A100 is faster).
2. The repo is private, so add a GitHub token once: click the 🔑 **Secrets** icon on the left, add `GITHUB_TOKEN` (a fine-grained token with *Contents: read and write* on this repo) and allow notebook access.
3. Run the cells top to bottom. Results go to Google Drive under `MyDrive/diploma-mer/`, so they survive when the runtime is reset.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/diploma-mer'   # shared results folder
import os
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/meld', exist_ok=True)

In [ ]:
# Clone (or update) the repo
REPO = 'NurimanB/multimodal-emotion-recognition'
BRANCH = 'main'

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/' + REPO.split('/')[1]

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Colab already has torch and transformers; this adds the rest and installs the package
!pip install -q -r requirements.txt
!pip install -q -e .
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## Data
The MELD text CSVs are small (~1.5 MB) and are stored on Drive.

In [ ]:
!python scripts/download_meld.py --data_dir {DRIVE_DIR}/data/meld

## 1. Smoke test (~3 min)
xlm-roberta-base on 1000 examples, 1 epoch. Only checks that everything runs; the score is meaningless.

In [ ]:
!python -m mer.text.train --config configs/smoke.yaml data_dir={DRIVE_DIR}/data/meld output_dir={DRIVE_DIR}/outputs/smoke

## 2. Main run: XLM-RoBERTa-large + 3 turns of dialogue context

About 1–1.5 h on a T4 (less on L4/A100). Change settings by adding `key=value`, e.g. `n_context=5`, `lr=2e-5`, `epochs=8`.
If a T4 runs out of memory, add `batch_size=4 grad_accum=4` or `gradient_checkpointing=true`.

In [ ]:
RUN = 'text_xlmr_large_ctx3'
!python -m mer.text.train --config configs/text_xlmr_large.yaml data_dir={DRIVE_DIR}/data/meld output_dir={DRIVE_DIR}/outputs/{RUN}

## 3. English reference: RoBERTa-large (optional)

In [ ]:
RUN = 'text_roberta_large_ctx3'
!python -m mer.text.train --config configs/text_roberta_large.yaml data_dir={DRIVE_DIR}/data/meld output_dir={DRIVE_DIR}/outputs/{RUN}

## Results
Weighted F1 is the main metric. Log each run in Notion → Experiments.

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{DRIVE_DIR}/outputs/*/results.json')):
    r = json.load(open(f))
    rows.append({'run': f.split('/')[-2],
                 'dev_wF1': round(r['dev']['weighted_f1'], 4), 'test_wF1': round(r['test']['weighted_f1'], 4),
                 'test_macroF1': round(r['test']['macro_f1'], 4), 'test_acc': round(r['test']['accuracy'], 4),
                 'minutes': r.get('train_minutes')})
pd.DataFrame(rows)